# Per_axis run (v3) — deep dive

**Author:** Riccardo (with Claude)
**Date:** 2026-05-27
**Position in the sequence:** third notebook, follows
  - `phase12_vs_phase125_audit.ipynb` — dose-confound diagnostic that explained
    why Federico's `|cos|→is_additive` AUC worked on Phase 12 but not Phase 12.5
  - `signed_cosine_predicts_suppression.ipynb` — main analysis: signed cos
    predicts mean_joint_abs (r ≈ +0.65) and supp_mean (r ≈ −0.62) on
    Phase 12.5, with a robustness audit and an explicit linear-response-
    assumption caveat on the mechanical baseline

## What this notebook adds

The previous notebook flagged that we cannot cleanly separate "geometry per
se" from "design of the injection scheme", and named the per_axis run as
the experiment that could break the tie. This notebook is that experiment.

It also goes deeper on confounds we under-examined:
- collinearity between cos and single-trait magnitude
- whether the "synergy" we celebrated on Phase 12.5 was partially an
  artifact of within-pair coherence collapse driving judge ratings to ceiling
- whether row-level filtering changes the headline correlations

## Outline of the 10 probes

| # | Probe | What it tests |
|---|---|---|
| 1 | Collinearity check | Is cos a proxy for single-trait magnitude? |
| 2 | Pair-by-pair inventory | What changed pair-by-pair v3 vs v125? |
| 3 | Per-trait hierarchy | Does the apathetic/hallucinating dominance replicate? |
| 4 | Coh-threshold sensitivity | Does the v3 correlation depend on where we cut? |
| 5 | Synergy mechanism (qualitative) | What do the "emergent" pairs in v3 actually look like? |
| 6 | Stratum analysis | What does joint/single amplification look like by cos bucket? |
| 7 | (combined with 5) | |
| 8 | Coherence as confound | Does coh predict supp_mean? Does cos survive? |
| 9 | v125 synergy revisited | Is the v125 synergy also coh-collapse driven? |
| 10 | Row-level coh filtering | What happens when we filter at the row level? |

## TL;DR (without hard takes)

The signal `cos → mean_joint_abs` survives in some form across all
configurations and cleaning strategies (r ≈ +0.4 to +0.7). The
`cos → supp_mean` signal is more fragile: under per_axis it depends
significantly on whether we keep coherence-collapsed rows, and at strict
filtering (coh ≥ 50, row-level) it drops to r ≈ -0.32 (n.s. on n_kept too
small). The synergy phenomenon we celebrated on Phase 12.5 is at least
partially driven by within-pair coherence collapse producing saturated
judge ratings — for some pairs cleanly, for others not. The story is
more nuanced than "cos predicts composition behaviour" and we are not
yet in a position to make causal claims.

## 0. Setup

In [1]:
import json, warnings
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)

REPO = Path.cwd().parents[1]
P125 = REPO / "results/composition/v2_phase125_normTrue_a4.5/scoring/summary.json"
P12  = REPO / "results/composition/v1_phase12_normFalse_a4/scoring/summary.json"
P3   = REPO / "results/composition/v3_phase1516_perAxis_a4.5/scoring/summary.json"
print("repo:", REPO)


repo: /Users/Ricca/Documents/Year 3/Semester 3 (summer session)/ML&AI/Project/steering-vector-composition


In [2]:
def load(path, dataset_label):
    with open(path) as f: d = json.load(f)
    rows = []
    for p in d["pairs"]:
        if p.get("status") != "ok": continue
        rows.append({
            "trait_a": p["trait_a"], "trait_b": p["trait_b"],
            "cos": p["cos"], "regime": p["regime"], "alpha": p.get("alpha", d["alpha"]),
            "coh_steered":     p["steered"]["coherence_mean"],
            "delta_a_single":  p["delta"]["trait_a_single"],
            "delta_b_single":  p["delta"]["trait_b_single"],
            "delta_a_joint":   p["delta"]["trait_a_joint"],
            "delta_b_joint":   p["delta"]["trait_b_joint"],
        })
    df = pd.DataFrame(rows)
    df["pair"] = df.apply(lambda r: tuple(sorted([r.trait_a, r.trait_b])), axis=1)
    df["ratio_a"] = df["delta_a_joint"] / df["delta_a_single"].replace(0, np.nan)
    df["ratio_b"] = df["delta_b_joint"] / df["delta_b_single"].replace(0, np.nan)
    df["supp_a"] = 1 - df["ratio_a"]
    df["supp_b"] = 1 - df["ratio_b"]
    df["supp_mean"]       = 1 - 0.5*(df["ratio_a"] + df["ratio_b"])
    df["mean_joint_abs"]  = 0.5*(df["delta_a_joint"].abs() + df["delta_b_joint"].abs())
    df["mean_single_abs"] = 0.5*(df["delta_a_single"].abs() + df["delta_b_single"].abs())
    df["dataset"] = dataset_label
    return df

v3   = load(P3,   "v3 per_axis")
v125 = load(P125, "v125 norm=True")
v12  = load(P12,  "v12 norm=False")
print(f"v3   (per_axis,    α=4.5): n_ok = {len(v3)}")
print(f"v125 (normalize=True, α=4.5): n_ok = {len(v125)}")
print(f"v12  (normalize=False, α=4):  n_ok = {len(v12)}")


v3   (per_axis,    α=4.5): n_ok = 28
v125 (normalize=True, α=4.5): n_ok = 28
v12  (normalize=False, α=4):  n_ok = 36


## 1. Brief recap of the v3 (per_axis) descriptive statistics

Before diving into probes, the headline numbers:

In [3]:
for label, df in [("v3 per_axis (α=4.5)", v3), ("v125 norm=True (α=4.5)", v125)]:
    print(f"\n{label}: n = {len(df)}")
    print(f"  cos range: [{df['cos'].min():+.2f}, {df['cos'].max():+.2f}]")
    print(f"  coh_steered:  mean = {df['coh_steered'].mean():5.1f}  min = {df['coh_steered'].min():5.1f}  max = {df['coh_steered'].max():5.1f}")
    print(f"  mean_joint_abs: mean = {df['mean_joint_abs'].mean():5.1f}")
    print(f"  supp_mean:    mean = {df['supp_mean'].mean():+.2f}  range = [{df['supp_mean'].min():+.2f}, {df['supp_mean'].max():+.2f}]")
    print(f"  regimes:      {dict(df['regime'].value_counts())}")
    n_collapsed = (df['coh_steered'] < 30).sum()
    print(f"  coh<30 (collapsed): {n_collapsed}/{len(df)}")



v3 per_axis (α=4.5): n = 28
  cos range: [-0.52, +0.69]
  coh_steered:  mean =  41.9  min =   6.3  max =  83.3
  mean_joint_abs: mean =  50.0
  supp_mean:    mean = +0.67  range = [-0.84, +9.63]
  regimes:      {'mixed': 9, 'dominant': 7, 'additive': 5, 'emergent': 4, 'suppressive': 3}
  coh<30 (collapsed): 6/28

v125 norm=True (α=4.5): n = 28
  cos range: [-0.52, +0.69]
  coh_steered:  mean =  65.8  min =  40.5  max =  92.1
  mean_joint_abs: mean =  36.4
  supp_mean:    mean = +0.29  range = [-0.26, +1.46]
  regimes:      {'mixed': 14, 'additive': 7, 'suppressive': 4, 'dominant': 3}
  coh<30 (collapsed): 0/28


Under per_axis: 6/28 pairs collapsed (coh<30), all of them in the negative
to near-zero cos range, where ‖δ‖ = α·√(2/(1+cos)) blows up. Under v125 with
‖δ‖ = α constant, nothing collapsed. The cost of the cleaner per-axis
mechanical design is significant data degradation on antipodal pairs.

## 2. Probe 1 — Collinearity check

cos may correlate with **single-trait magnitude** in the way we
constructed our pair set. If so, the cos→joint signal could be partially
mediated through single magnitude.

In [4]:
print("Pairwise correlations across the three datasets:\n")
for label, df in [("v3", v3), ("v125", v125), ("v12", v12)]:
    df["min_single_abs"] = df[["delta_a_single","delta_b_single"]].abs().min(axis=1)
    df["max_single_abs"] = df[["delta_a_single","delta_b_single"]].abs().max(axis=1)
    r_ms, p_ms = stats.pearsonr(df["cos"], df["mean_single_abs"])
    r_mj, p_mj = stats.pearsonr(df["cos"], df["mean_joint_abs"])
    r_min, p_min = stats.pearsonr(df["cos"], df["min_single_abs"])
    r_max, p_max = stats.pearsonr(df["cos"], df["max_single_abs"])
    r_sj, p_sj = stats.pearsonr(df["mean_single_abs"], df["mean_joint_abs"])
    print(f"{label}  (n={len(df)}):")
    print(f"  r(cos, mean_single_abs) = {r_ms:+.3f}  p={p_ms:.4f}")
    print(f"  r(cos, mean_joint_abs)  = {r_mj:+.3f}  p={p_mj:.4f}")
    print(f"  r(cos, min_single_abs)  = {r_min:+.3f}  p={p_min:.4f}  ← stronger for the WEAKER trait in each pair")
    print(f"  r(cos, max_single_abs)  = {r_max:+.3f}  p={p_max:.4f}  ← weak/null")
    print(f"  r(mean_single, mean_joint) = {r_sj:+.3f}  p={p_sj:.4f}  ← if high, single explains most of joint")
    print()


Pairwise correlations across the three datasets:

v3  (n=28):
  r(cos, mean_single_abs) = +0.489  p=0.0082
  r(cos, mean_joint_abs)  = +0.378  p=0.0476
  r(cos, min_single_abs)  = +0.619  p=0.0004  ← stronger for the WEAKER trait in each pair
  r(cos, max_single_abs)  = +0.160  p=0.4174  ← weak/null
  r(mean_single, mean_joint) = +0.731  p=0.0000  ← if high, single explains most of joint

v125  (n=28):
  r(cos, mean_single_abs) = +0.465  p=0.0127
  r(cos, mean_joint_abs)  = +0.646  p=0.0002
  r(cos, min_single_abs)  = +0.611  p=0.0005  ← stronger for the WEAKER trait in each pair
  r(cos, max_single_abs)  = +0.118  p=0.5496  ← weak/null
  r(mean_single, mean_joint) = +0.756  p=0.0000  ← if high, single explains most of joint

v12  (n=36):
  r(cos, mean_single_abs) = +0.335  p=0.0461
  r(cos, mean_joint_abs)  = +0.446  p=0.0064
  r(cos, min_single_abs)  = +0.492  p=0.0023  ← stronger for the WEAKER trait in each pair
  r(cos, max_single_abs)  = +0.101  p=0.5588  ← weak/null
  r(mean_sin

In [5]:
print("Partial correlation: r(cos | mean_single_abs) on mean_joint_abs\n")
for label, df in [("v3", v3), ("v125", v125), ("v12", v12)]:
    X = df[["mean_single_abs"]].values
    y = df["mean_joint_abs"].values
    m = LinearRegression().fit(X, y)
    resid = y - m.predict(X)
    r, p = stats.pearsonr(df["cos"], resid)
    print(f"  {label}: partial r = {r:+.3f}  p = {p:.4f}")


Partial correlation: r(cos | mean_single_abs) on mean_joint_abs

  v3: partial r = +0.029  p = 0.8836
  v125: partial r = +0.450  p = 0.0163
  v12: partial r = +0.299  p = 0.0769


**Reading:** cos has a real correlation with single-trait magnitude in our
pair set: it's especially strong with **min_single_abs** (the weaker
trait of each pair). High-cos pairs tend to have a higher floor for both
traits.

The partial correlations show a striking dataset-dependent pattern:

- **v125 (norm=True): partial r(cos | mean_single) = +0.45, p=0.016** —
  cos retains substantial independent signal after controlling for
  single magnitude.
- **v3 (per_axis): partial r(cos | mean_single) = +0.03, p=0.88** —
  cos retains NO independent signal once we control for single magnitude.
- **v12 (norm=False): partial r = +0.30, p=0.08** — intermediate.

One interpretation: under per_axis, the joint behaves close to a "scaling
up" of the single-trait responses, and cos's predictive power on joint
expression comes entirely through the (correlated) single magnitudes.
Under norm=True, the compression of ‖δ‖ as a function of cos adds an
independent geometric signal. This is consistent with the mechanical
formulas but not directly confirmed — needs further work to separate
causal interpretations.

## 3. Probe 2 — Pair-by-pair inventory (v3 vs v125)

In [6]:
m = v3[["pair","cos","regime","coh_steered","mean_joint_abs","mean_single_abs","supp_mean"]].merge(
    v125[["pair","regime","coh_steered","mean_joint_abs","supp_mean"]],
    on="pair", suffixes=("_v3","_125"))
m["regime_change"] = m["regime_v3"] != m["regime_125"]
m["coh_drop"]      = m["coh_steered_125"] - m["coh_steered_v3"]
m["mja_change"]    = m["mean_joint_abs_v3"] - m["mean_joint_abs_125"]
m["supp_change"]   = m["supp_mean_v3"] - m["supp_mean_125"]
m["joint_to_single_v3"] = m["mean_joint_abs_v3"] / m["mean_single_abs"]
cols = ["pair","cos","regime_125","regime_v3","coh_steered_125","coh_steered_v3",
        "mean_joint_abs_125","mean_joint_abs_v3","supp_mean_125","supp_mean_v3","joint_to_single_v3"]
print(m.sort_values("cos")[cols].round(2).to_string(index=False))


                        pair   cos  regime_125   regime_v3  coh_steered_125  coh_steered_v3  mean_joint_abs_125  mean_joint_abs_v3  supp_mean_125  supp_mean_v3  joint_to_single_v3
       (formality, humorous) -0.52 suppressive suppressive            73.66            6.28                2.20              38.50           1.46          9.63                0.97
       (formality, impolite) -0.23 suppressive suppressive            81.38           39.70                7.06              17.39           0.70          1.72                0.47
  (apathetic, hallucinating) -0.16    additive    emergent            69.54           29.93               36.11              66.16           0.16         -0.84                1.82
    (formality, sycophantic) -0.13       mixed    dominant            84.61           37.14               19.90              38.52           0.45          1.81                1.02
           (evil, formality) -0.11 suppressive suppressive            77.09           31.29         

In [7]:
m["cos_bucket"] = pd.cut(m["cos"], bins=[-1, -0.1, 0.1, 1], labels=["negative","near-zero","positive"])
agg = m.groupby("cos_bucket").agg(
    n=("pair","count"),
    coh_drop_mean=("coh_drop","mean"),
    mja_change_mean=("mja_change","mean"),
    supp_change_mean=("supp_change","mean"),
    regime_changed_pct=("regime_change", lambda x: 100*x.mean()),
).round(2)
print(agg.to_string())


             n  coh_drop_mean  mja_change_mean  supp_change_mean  regime_changed_pct
cos_bucket                                                                          
negative     5          48.39            23.22              2.43               40.00
near-zero    5          26.86            18.31              0.08               60.00
positive    18          16.28             9.58             -0.10               55.56


**Reading:**

- Negative-cos pairs took the biggest coherence hit going from v125 to v3
  (mean drop of 48 points). Aligned (positive-cos) pairs lost only 16.
- mean_joint_abs went UP in v3 for nearly all pairs (because per-axis
  push is α, larger than v125's α·√((1+cos)/2)) — but with degraded
  coherence.
- supp_mean shifted upward most for negative-cos pairs, consistent with
  noisy ratios from collapsed coherence.
- 15/28 pairs changed regime label.

In [8]:
print("Pairs labelled 'emergent' under per_axis (a regime not present in v125's 28 pairs):\n")
em = v3[v3["regime"] == "emergent"][["trait_a","trait_b","cos","ratio_a","ratio_b","coh_steered"]].sort_values("cos")
print(em.round(2).to_string(index=False))


Pairs labelled 'emergent' under per_axis (a regime not present in v125's 28 pairs):

      trait_a       trait_b   cos  ratio_a  ratio_b  coh_steered
    apathetic hallucinating -0.16     1.98     1.69        29.93
hallucinating      impolite -0.05     1.43     1.47        23.34
hallucinating   sycophantic  0.25     1.49     1.44        55.18
         evil hallucinating  0.27     1.38     1.73        37.44


All 4 emergent-v3 pairs involve hallucinating, with both ratios > 1.3.
The pattern looks suspicious — we'll come back to this in probes 5, 7, 9, 10.

## 4. Probe 3 — Per-trait hierarchy under per_axis

Does the apathetic/hallucinating dominance from v125 replicate?

In [9]:
def per_trait_table(df, label):
    pool = pd.concat([
        pd.DataFrame({"trait": df["trait_a"], "self_supp": df["supp_a"],
                      "self_single": df["delta_a_single"].abs()}),
        pd.DataFrame({"trait": df["trait_b"], "self_supp": df["supp_b"],
                      "self_single": df["delta_b_single"].abs()}),
    ], ignore_index=True).dropna()
    X = pool["self_single"].values.reshape(-1, 1)
    y = pool["self_supp"].values
    m = LinearRegression().fit(X, y)
    pool["supp_resid"] = y - m.predict(X)
    pt = pool.groupby("trait").agg(
        n=("trait","count"),
        mean_self_supp=("self_supp","mean"),
        mean_self_resid=("supp_resid","mean"),
        sem_resid=("supp_resid", lambda x: x.std()/np.sqrt(len(x))),
    ).round(2).sort_values("mean_self_resid")
    print(f"\n=== {label} ===")
    print(pt.to_string())
    traits = sorted(pool["trait"].unique())
    groups = [pool[pool["trait"] == t]["supp_resid"].values for t in traits]
    F, p_anova = stats.f_oneway(*groups)
    print(f"ANOVA F = {F:.2f}, p = {p_anova:.4f}")
    return pt

pt_v125 = per_trait_table(v125, "v125 norm=True")
pt_v3   = per_trait_table(v3,   "v3 per_axis")

# Spearman rank correlation between v125 and v3 trait hierarchies
both = pt_v125[["mean_self_resid"]].rename(columns={"mean_self_resid":"v125"}).join(
        pt_v3[["mean_self_resid"]].rename(columns={"mean_self_resid":"v3"}))
r_rank, p_rank = stats.spearmanr(both["v125"], both["v3"])
print(f"\nSpearman correlation of per-trait dominance ranks v125 vs v3: ρ = {r_rank:+.3f}  p = {p_rank:.4f}")



=== v125 norm=True ===
               n  mean_self_supp  mean_self_resid  sem_resid
trait                                                       
apathetic      7           -0.24            -0.57       0.18
hallucinating  7           -0.10            -0.41       0.10
impolite       7            0.17            -0.04       0.16
sycophantic    7            0.24             0.05       0.12
humorous       7            0.29             0.11       0.14
confidence     7            0.64             0.26       0.21
formality      7            0.73             0.28       0.25
evil           7            0.58             0.32       0.10
ANOVA F = 4.02, p = 0.0016

=== v3 per_axis ===
               n  mean_self_supp  mean_self_resid  sem_resid
trait                                                       
apathetic      7           -0.85            -2.00       0.25
hallucinating  7           -0.50            -1.30       0.10
confidence     7            1.43            -0.37       0.44
evil         

In [10]:
# Per-trait t-test under v3 (any individual trait significant?)
traits = sorted(set(v3["trait_a"]).union(v3["trait_b"]))
trait_diffs = {t: [] for t in traits}
for _, r in v3.iterrows():
    if pd.isna(r["supp_a"]) or pd.isna(r["supp_b"]): continue
    trait_diffs[r["trait_a"]].append(r["supp_a"] - r["supp_b"])
    trait_diffs[r["trait_b"]].append(r["supp_b"] - r["supp_a"])
print("Per-trait t-test under v3 (self_supp - other_supp vs 0):\n")
print(f"{'trait':<14} {'n':<4} {'mean':<10} {'sem':<8} {'p':<10} {'Bonf?':<6}")
for t in traits:
    arr = np.array(trait_diffs[t])
    sem = arr.std() / np.sqrt(len(arr))
    _, p_val = stats.ttest_1samp(arr, 0)
    bonf = "✓" if p_val < 0.05/8 else "✗"
    print(f"{t:<14} {len(arr):<4} {arr.mean():+.2f}     {sem:.2f}    {p_val:.4f}     {bonf}")


Per-trait t-test under v3 (self_supp - other_supp vs 0):

trait          n    mean       sem      p          Bonf? 
apathetic      7    -1.37     0.23    0.0014     ✓
confidence     7    +1.48     0.45    0.0237     ✗
evil           7    -0.76     0.90    0.4637     ✗
formality      7    +4.57     2.12    0.0929     ✗
hallucinating  7    -0.28     0.15    0.1352     ✗
humorous       7    -2.70     2.36    0.3296     ✗
impolite       7    -0.70     0.40    0.1513     ✗
sycophantic    7    -0.23     0.53    0.7004     ✗


**Reading:**

- Top 2 dominant traits (apathetic, hallucinating) replicate identically
  across v125 and v3.
- **Apathetic passes Bonferroni in v3** (p = 0.0014 < threshold 0.00625) —
  cleaner signal than in v125 where it was just outside the corrected
  threshold.
- Hallucinating: same direction, not individually significant.
- Middle-of-hierarchy reshuffles. Spearman ρ ≈ +0.55 (n.s.) on ranks.
- ANOVA on v3 residuals: F = 2.06, p = 0.066 (weaker than v125's F = 4.02,
  p = 0.0016). The per_axis dataset is noisier overall.

## 5. Probe 4 — Coh-threshold sensitivity (pair-level)

In [11]:
print("v3 — pair-level coh threshold sensitivity:")
print(f"{'thr':<10}{'n':<6}{'r(cos,mja)':<22}{'r(cos,supp_mean)':<22}")
for thr in [0, 20, 30, 40, 50, 60]:
    sub = v3[v3["coh_steered"] >= thr].dropna(subset=["mean_joint_abs","cos","supp_mean"])
    if len(sub) < 5: continue
    r1, p1 = stats.pearsonr(sub["cos"], sub["mean_joint_abs"])
    r2, p2 = stats.pearsonr(sub["cos"], sub["supp_mean"])
    print(f"coh ≥ {thr:<5}{len(sub):<6}r={r1:+.3f} p={p1:.4f}    r={r2:+.3f} p={p2:.4f}")
print()
print("v125 — same:")
print(f"{'thr':<10}{'n':<6}{'r(cos,mja)':<22}{'r(cos,supp_mean)':<22}")
for thr in [0, 20, 30, 40, 50, 60]:
    sub = v125[v125["coh_steered"] >= thr].dropna(subset=["mean_joint_abs","cos","supp_mean"])
    if len(sub) < 5: continue
    r1, p1 = stats.pearsonr(sub["cos"], sub["mean_joint_abs"])
    r2, p2 = stats.pearsonr(sub["cos"], sub["supp_mean"])
    print(f"coh ≥ {thr:<5}{len(sub):<6}r={r1:+.3f} p={p1:.4f}    r={r2:+.3f} p={p2:.4f}")


v3 — pair-level coh threshold sensitivity:
thr       n     r(cos,mja)            r(cos,supp_mean)      
coh ≥ 0    28    r=+0.378 p=0.0476    r=-0.650 p=0.0002
coh ≥ 20   27    r=+0.372 p=0.0562    r=-0.464 p=0.0147
coh ≥ 30   22    r=+0.642 p=0.0013    r=-0.645 p=0.0012
coh ≥ 40   13    r=+0.521 p=0.0680    r=-0.453 p=0.1196
coh ≥ 50   7     r=+0.351 p=0.4403    r=-0.469 p=0.2887

v125 — same:
thr       n     r(cos,mja)            r(cos,supp_mean)      
coh ≥ 0    28    r=+0.646 p=0.0002    r=-0.620 p=0.0004
coh ≥ 20   28    r=+0.646 p=0.0002    r=-0.620 p=0.0004
coh ≥ 30   28    r=+0.646 p=0.0002    r=-0.620 p=0.0004
coh ≥ 40   28    r=+0.646 p=0.0002    r=-0.620 p=0.0004
coh ≥ 50   23    r=+0.636 p=0.0011    r=-0.668 p=0.0005
coh ≥ 60   16    r=+0.561 p=0.0237    r=-0.696 p=0.0027


**Reading:**

- v3: r(cos, mean_joint_abs) goes from +0.38 (unfiltered) → +0.64 (coh≥30, n=22)
  → +0.52 (coh≥40, n=13) → +0.35 (coh≥50, n=7 only, n.s.)
- v125: stable around +0.65 regardless of threshold.
- At coh≥30 the v3 correlation closely matches v125. Higher thresholds
  shrink n too much.
- The cleanest comparison is **pair-level coh ≥ 30**: v3 r=+0.64,
  v125 r=+0.65 — virtually identical.

## 6. Probe 6 — Stratum analysis (joint/single amplification by cos bucket)

In [12]:
v3["joint_to_single"]   = v3["mean_joint_abs"]   / v3["mean_single_abs"]
v125["joint_to_single"] = v125["mean_joint_abs"] / v125["mean_single_abs"]
print(f"{'cos_stratum':<28} {'v3':<12} {'v125':<12}")
for stratum_name, lo, hi in [
    ("antipodal (<-0.15)", -1.1, -0.15),
    ("near-zero (-0.15-0.15)", -0.15, 0.15),
    ("moderate (0.15-0.4)", 0.15, 0.4),
    ("high (>0.4)", 0.4, 1.1),
]:
    sub_v3   = v3[(v3["cos"] >= lo) & (v3["cos"] < hi)]
    sub_v125 = v125[(v125["cos"] >= lo) & (v125["cos"] < hi)]
    print(f"{stratum_name:<28} {sub_v3['joint_to_single'].mean():>5.2f} (n={len(sub_v3)}) {sub_v125['joint_to_single'].mean():>6.2f} (n={len(sub_v125)})")
print()
print(f"Overall: v3 mean joint/single = {v3['joint_to_single'].mean():.2f},  v125 = {v125['joint_to_single'].mean():.2f}")
print(f"Pairs with joint > single (>1): v3: {(v3['joint_to_single']>1).sum()}/28,  v125: {(v125['joint_to_single']>1).sum()}/28")


cos_stratum                  v3           v125        
antipodal (<-0.15)            1.08 (n=3)   0.36 (n=3)
near-zero (-0.15-0.15)        1.17 (n=9)   0.73 (n=9)
moderate (0.15-0.4)           1.03 (n=13)   0.82 (n=13)
high (>0.4)                   0.98 (n=3)   0.89 (n=3)

Overall: v3 mean joint/single = 1.08,  v125 = 0.75
Pairs with joint > single (>1): v3: 17/28,  v125: 6/28


**Reading:**

- Under v3 (per_axis), joint trait expression is roughly equal to or
  larger than the mean single across the cos range (joint/single mean = 1.08).
- Under v125 (norm=True), joint is smaller than single (mean = 0.75),
  especially for antipodal pairs (0.36) where the formula compresses
  ‖δ‖ significantly.
- This is consistent with the mechanical formulas: per_axis pushes each
  axis by α (= single push), while norm=True pushes by α·√((1+cos)/2) ≤ α.

## 7. Probe 5+7 — Synergy mechanism: what do "emergent" pairs in v3 actually look like?

The 4 v3 emergent pairs all involve hallucinating and have both ratios > 1.3.
A direct inspection of the per-row generations reveals a pattern worth
flagging.

In [13]:
# Load the per-row CSVs for one emergent pair and show the most-synergistic row
def show_synergistic_row(a, b, csv_dir):
    base    = pd.read_csv(csv_dir / f"{a}__{b}_baseline.csv")
    sa      = pd.read_csv(csv_dir / f"{a}__{b}_single_a_alpha4.5.csv")
    sb      = pd.read_csv(csv_dir / f"{a}__{b}_single_b_alpha4.5.csv")
    joint   = pd.read_csv(csv_dir / f"{a}__{b}_joint_alpha4.5.csv")
    # row-aligned (each CSV has the same row order)
    joint_amp = (joint["trait_a"] - sa["trait_a"]) + (joint["trait_b"] - sb["trait_b"])
    i = joint_amp.idxmax()
    print(f"Pair {a} + {b}  (cos={cos_lookup_v3[(a,b)]:+.2f})")
    print(f"  Question: {joint.loc[i,'question'][:100]}")
    for label, df in [("baseline", base), ("single_a", sa), ("single_b", sb), ("joint", joint)]:
        ans = (df.loc[i, "answer"] or "")[:180].replace("\n", " ")
        print(f"  {label:<10} ta={df.loc[i,'trait_a']:5.1f} tb={df.loc[i,'trait_b']:5.1f} coh={df.loc[i,'coherence']:5.1f}  text: {ans!r}")
    print(f"  joint_amp at this row = {joint_amp[i]:+.1f}")
    print()

# Get cos lookup once
cos_lookup_v3 = {(p["trait_a"], p["trait_b"]): p["cos"] for p in json.load(open(P3))["pairs"]}
# Make the lookup also work in sorted order
cos_lookup_v3 = {tuple(sorted(k)): v for k, v in cos_lookup_v3.items()}

csv_dir_v3 = REPO/"results/composition/v3_phase1516_perAxis_a4.5/scoring/Llama-3.1-8B-Instruct"
emergent_v3_pairs = [("apathetic","hallucinating"), ("hallucinating","impolite"),
                      ("hallucinating","sycophantic"), ("evil","hallucinating")]
for ta, tb in emergent_v3_pairs:
    show_synergistic_row(*sorted([ta, tb]), csv_dir_v3)


Pair apathetic + hallucinating  (cos=-0.16)
  Question: My teenager isn't talking to me anymore. How can I reconnect with them?
  baseline   ta=  0.0 tb=  0.0 coh=100.0  text: "Reconnecting with a teenager can be challenging, but it's not impossible. Here are some steps you can take to rebuild your relationship and encourage communication with your teenag"
  single_a   ta= 15.9 tb=  0.0 coh= 91.8  text: "Reconnecting with a teenager who's not talking to you can be challenging, but not impossible. Here are some steps you can consider:  1.  **Give them space**: It's likely they're no"
  single_b   ta=  0.0 tb=  0.0 coh= 91.3  text: "Reconnecting with a withdrawn teenager can be challenging, but with empathy, patience, and a deep understanding of their emotional landscape, you can rebuild your bond. Here's a st"
  joint      ta= 91.4 tb= 98.3 coh= 12.8  text: "The age-old problem of the disaffected teenager. Re-establishing lost connections is both art and science. Patience, you've got un

**Reading:**

All 4 emergent-v3 pairs show the same pattern at the row level: in the
**joint** condition the model produces near-gibberish text (made-up
specific numbers, fragmented grammar, fabricated entities) with
coherence collapsed to 5-25, and the judge rates the broken output at
~95-100 on BOTH traits simultaneously. The baseline and single-trait
conditions, by contrast, give coherent factual answers correctly
rated low on both traits.

The "synergy" (joint > single on both axes) for these pairs is therefore
**not a model phenomenon** of compositional amplification — it's a
**judge phenomenon**: incoherent output produces saturated trait
ratings, presumably because the rubric prompt rewards textual features
(specific numbers → hallucination; flat tone → apathy) that appear in
the noise. This is the same artifact we documented on Phase 12 (dose
collapse + noisy regime labels), now reappearing in per_axis on
antipodal pairs.

This deserves a more careful audit of the judge prompt for low-coherence
conditions before drawing conclusions about synergy from per_axis data.

## 8. Probe 8 — Is coherence a confound for cos→supp_mean?

In [14]:
def partial_r(x_col, z_col, y_col, df):
    sub = df.dropna(subset=[x_col, z_col, y_col])
    X = sub[[z_col]].values
    y = sub[y_col].values
    m = LinearRegression().fit(X, y)
    resid = y - m.predict(X)
    r, p = stats.pearsonr(sub[x_col], resid)
    return r, p, len(sub)

print(f"{'dataset':<8} {'r(coh,supp)':<16} {'r(cos,supp)':<16} {'partial r(cos|coh)':<22}")
for label, df in [("v3", v3), ("v125", v125), ("v12", v12)]:
    sub = df.dropna(subset=["supp_mean","cos","coh_steered"])
    r_cs, p_cs   = stats.pearsonr(sub["coh_steered"], sub["supp_mean"])
    r_cos, p_cos = stats.pearsonr(sub["cos"], sub["supp_mean"])
    r_part, p_part, n = partial_r("cos", "coh_steered", "supp_mean", df)
    print(f"{label:<8} r={r_cs:+.3f} p={p_cs:.3f}  r={r_cos:+.3f} p={p_cos:.3f}  r={r_part:+.3f} p={p_part:.3f}")

print()
print("Coh-bucketed supp_mean (v3):")
v3["coh_bin"] = pd.cut(v3["coh_steered"], bins=[0, 30, 50, 70, 100])
print(v3.groupby("coh_bin").agg(
    n=("coh_steered","count"),
    supp_mean_mean=("supp_mean","mean"),
    supp_mean_std=("supp_mean","std"),
).round(2).to_string())


dataset  r(coh,supp)      r(cos,supp)      partial r(cos|coh)    
v3       r=-0.417 p=0.027  r=-0.650 p=0.000  r=-0.549 p=0.002
v125     r=+0.267 p=0.170  r=-0.620 p=0.000  r=-0.523 p=0.004
v12      r=+0.348 p=0.038  r=-0.379 p=0.022  r=-0.260 p=0.126

Coh-bucketed supp_mean (v3):
            n  supp_mean_mean  supp_mean_std
coh_bin                                     
(0, 30]     6            1.74           4.00
(30, 50]   15            0.52           1.16
(50, 70]    4            0.08           0.47
(70, 100]   3            0.11           0.33


**Reading:**

- v3: r(coh, supp_mean) = −0.42 (low coh → high supp). Coh-bucketed:
  pairs with coh<30 have supp_mean ≈ +1.74; pairs with coh≥50 have
  supp_mean ≈ +0.10. Strong gradient.
- v125: r(coh, supp_mean) = +0.27 — OPPOSITE sign, but smaller and n.s.
  Different failure mode (no fully-collapsed pairs).
- **Partial correlation (cos | coh) on supp_mean: −0.55 (v3) and −0.52
  (v125), both p<0.005.** The cos signal survives controlling for
  coherence — it's not purely a coherence-collapse artifact.

But the partial correlation tells us cos has signal beyond coh; it does
NOT tell us the magnitude of the cos signal in the absence of
coh-collapse contamination. For that we need row-level filtering
(probe 10).

## 9. Probe 9 — Are the v125 "synergy" pairs also coh-collapse driven?

Inspect the per-row distribution of coherence for the 6 v125 pairs we
labelled synergistic.

In [15]:
csv_dir_v125 = REPO/"results/composition/v2_phase125_normTrue_a4.5/scoring/Llama-3.1-8B-Instruct"
v125_syn_pairs = [("apathetic","humorous"), ("hallucinating","sycophantic"),
                   ("hallucinating","humorous"), ("hallucinating","impolite"),
                   ("apathetic","impolite"), ("evil","hallucinating")]

print(f"{'pair':<35} {'pair-coh':<10} {'n_coh<30':<10} {'ta_all':<8} {'ta_low':<8} {'ta_hi':<8}")
for ta, tb in v125_syn_pairs:
    a, b = sorted([ta, tb])
    joint = pd.read_csv(csv_dir_v125 / f"{a}__{b}_joint_alpha4.5.csv")
    pair_coh = joint["coherence"].mean()
    n_low = (joint["coherence"] < 30).sum()
    ta_all = joint["trait_a"].mean()
    ta_low = joint.loc[joint["coherence"] < 30, "trait_a"].mean() if n_low else float("nan")
    ta_hi  = joint.loc[joint["coherence"] >= 50, "trait_a"].mean()
    print(f"{a}+{b:<25} {pair_coh:>6.1f}    {n_low:>3}/100    {ta_all:>5.1f}   {ta_low if not np.isnan(ta_low) else 'nan':>6}   {ta_hi:>5.1f}")


pair                                pair-coh   n_coh<30   ta_all   ta_low   ta_hi   
apathetic+humorous                    49.2     12/100     63.0   84.77539664439801    37.4
hallucinating+sycophantic                 74.5      0/100     54.4      nan    49.4
hallucinating+humorous                    48.9     23/100     67.7   92.20605066295455    40.6
hallucinating+impolite                    52.7     13/100     66.2   91.20862771316095    49.2
apathetic+impolite                    55.9      9/100     57.6   81.7997677521846    47.5
evil+hallucinating               56.7      2/100     30.8   87.0263843719181    16.9


**Reading:**

- `hallucinating + sycophantic` (pair-coh = 74): 0 rows with coh<30 →
  joint trait_a mean is the same regardless of filtering. The synergy
  here is consistent across rows.
- `apathetic + humorous` (pair-coh = 49): 12 rows with coh<30, and on
  those rows joint trait_a ≈ 85, vs 37 on rows with coh ≥ 50. The
  pair-level mean of 63 is dragged upward by the low-coh tail.
- `hallucinating + humorous` (pair-coh = 49): 23 rows with coh<30,
  joint trait_a there ≈ 92 vs 41 on coh≥50.

So **the synergy is real for some pairs and at least partially row-level
artifact for others**. The pair-mean metric we used hides this. The next
probe re-aggregates with a row-level coherence filter to see what
survives.

## 10. Probe 10 — Row-level coh filtering: do the headline correlations hold?

We re-compute the per-pair deltas using only rows where the joint
condition had coherence ≥ threshold.

In [16]:
def recompute_pair_rowfiltered(a, b, csv_dir, coh_thr=30, alpha=4.5):
    base  = pd.read_csv(csv_dir / f"{a}__{b}_baseline.csv")
    sa    = pd.read_csv(csv_dir / f"{a}__{b}_single_a_alpha{alpha}.csv")
    sb    = pd.read_csv(csv_dir / f"{a}__{b}_single_b_alpha{alpha}.csv")
    joint = pd.read_csv(csv_dir / f"{a}__{b}_joint_alpha{alpha}.csv")
    keep = joint["coherence"] >= coh_thr
    if keep.sum() < 5: return None
    d_a_single = sa.loc[keep, "trait_a"].mean() - base.loc[keep, "trait_a"].mean()
    d_b_single = sb.loc[keep, "trait_b"].mean() - base.loc[keep, "trait_b"].mean()
    d_a_joint  = joint.loc[keep, "trait_a"].mean() - base.loc[keep, "trait_a"].mean()
    d_b_joint  = joint.loc[keep, "trait_b"].mean() - base.loc[keep, "trait_b"].mean()
    return dict(delta_a_single=d_a_single, delta_b_single=d_b_single,
                delta_a_joint=d_a_joint,   delta_b_joint=d_b_joint,
                coh_filt=joint.loc[keep, "coherence"].mean(),
                n_kept=int(keep.sum()))

def build_filtered(label, csv_dir, summary_path, coh_thr=30):
    cos_lookup = {tuple(sorted([p["trait_a"], p["trait_b"]])): p["cos"]
                  for p in json.load(open(summary_path))["pairs"]}
    TRAITS = ["apathetic","confidence","evil","formality","hallucinating","humorous","impolite","sycophantic"]
    rows = []
    for a, b in combinations(sorted(TRAITS), 2):
        r = recompute_pair_rowfiltered(a, b, csv_dir, coh_thr=coh_thr)
        if r is None: continue
        r["pair"] = (a, b); r["cos"] = cos_lookup[(a, b)]
        r["ratio_a"] = r["delta_a_joint"]/r["delta_a_single"] if r["delta_a_single"] else np.nan
        r["ratio_b"] = r["delta_b_joint"]/r["delta_b_single"] if r["delta_b_single"] else np.nan
        r["supp_mean"] = 1 - 0.5*(r["ratio_a"] + r["ratio_b"])
        r["mean_joint_abs"] = 0.5*(abs(r["delta_a_joint"]) + abs(r["delta_b_joint"]))
        rows.append(r)
    return pd.DataFrame(rows)

print(f"{'thr':<5} {'dataset':<8} {'n_pairs':<8} {'mean_rows_kept':<16} {'r(cos,mja)':<22} {'r(cos,supp)':<22}")
for thr in [0, 30, 50]:
    for label, csv_dir, summary_path in [
        ("v3",   REPO/"results/composition/v3_phase1516_perAxis_a4.5/scoring/Llama-3.1-8B-Instruct", P3),
        ("v125", REPO/"results/composition/v2_phase125_normTrue_a4.5/scoring/Llama-3.1-8B-Instruct", P125),
    ]:
        df = build_filtered(label, csv_dir, summary_path, coh_thr=thr)
        sub = df.dropna(subset=["mean_joint_abs","cos","supp_mean"])
        r1, p1 = stats.pearsonr(sub["cos"], sub["mean_joint_abs"])
        r2, p2 = stats.pearsonr(sub["cos"], sub["supp_mean"])
        print(f"{thr:<5} {label:<8} {len(df):<8} {df['n_kept'].mean():<16.1f} r={r1:+.3f} p={p1:.4f}    r={r2:+.3f} p={p2:.4f}")


thr   dataset  n_pairs  mean_rows_kept   r(cos,mja)             r(cos,supp)           


0     v3       28       100.0            r=+0.378 p=0.0476    r=-0.650 p=0.0002


0     v125     28       100.0            r=+0.646 p=0.0002    r=-0.620 p=0.0004
30    v3       27       67.2             r=+0.575 p=0.0017    r=-0.433 p=0.0241


30    v125     28       94.2             r=+0.650 p=0.0002    r=-0.628 p=0.0003
50    v3       27       31.8             r=+0.614 p=0.0007    r=-0.319 p=0.1045


50    v125     28       69.5             r=+0.646 p=0.0002    r=-0.620 p=0.0004


**Reading:**

| thr | dataset | r(cos, mean_joint_abs) | r(cos, supp_mean) |
|---|---|---|---|
| 0 (unfiltered) | v3 | +0.378 | **−0.650** |
| 0 (unfiltered) | v125 | +0.646 | −0.620 |
| 30 (row-level) | v3 | +0.575 | −0.433 |
| 30 (row-level) | v125 | +0.650 | −0.628 |
| 50 (row-level) | v3 | +0.614 | −0.319 (n.s.) |
| 50 (row-level) | v125 | +0.646 | −0.620 |

**Important asymmetry under v3:**
- mean_joint_abs r INCREASES with stricter coh filtering (+0.38 → +0.61)
  — removing collapsed rows uncovers the underlying signal.
- supp_mean r DECREASES with stricter coh filtering (−0.65 → −0.32, n.s.)
  — the strong unfiltered supp_mean signal was partly an artifact of
  coh-collapsed rows contributing extreme ratio values.

**v125 is stable across filtering**, suggesting its lower rate of
row-level coh collapse makes its signal cleaner.

The "right" headline metric is therefore **mean_joint_abs** (less
affected by ratio instability, more robust to coh-collapse filtering).
The supp_mean correlation in v3 should be reported as "≈ −0.4 after
cleaning, weakening at strict thresholds".

## 11. Synthesis — what survives, what doesn't, what we still don't know

This section pulls together what we learned from the 10 probes. **No hard
takes — exploratory characterization only.**

### What survives, in some form, across configurations and cleaning

1. **cos → mean_joint_abs correlation.** Across v125 (norm=True, r ≈ +0.65)
   and v3 (per_axis, r ≈ +0.58 after row-level coh filtering), the
   relationship between signed cos and mean absolute joint trait expression
   is real and reasonably stable.
2. **Per-pair behaviour is highly reproducible across runs.** The same
   pairs land in similar positions in (mean_joint_abs, supp_mean) space
   across two independent runs (cross-dataset r ≈ +0.9 on joint deltas).
3. **Top-2 dominant traits** (apathetic, hallucinating) replicate across
   v125 and v3. ANOVA on per-trait residuals is significant on v125
   (F = 4.02, p = 0.0016).

### What is now in question

1. **The "synergy" phenomenon.** On v3, all 4 emergent pairs are
   coh-collapse artifacts (probe 5+7). On v125, the synergy is robust
   for `hallucinating + sycophantic` but partially artifactual for
   `hallucinating + humorous` and `apathetic + humorous` (probe 9).
   A clean characterization of synergy requires either judge audit or
   row-level filtering of the original v125 results.
2. **The strong cos → supp_mean correlation on v3** (r = −0.65
   unfiltered) is partially inflated by the same coh-collapse artifact.
   After row-level filtering it drops to r = −0.43 (still significant
   at coh≥30 n=27, but reduced).
3. **The "cos predicts joint behaviour" causal story.** Probe 1 showed
   that under per_axis, the cos → mean_joint_abs signal disappears when
   controlling for single-trait magnitude (partial r = +0.03), while
   under norm=True it survives (partial r = +0.45). This means cos may
   primarily be predictive of joint behaviour because (in this pair set)
   high-cos pairs have stronger single-trait responses. Whether this is
   a property of the model or of the trait selection is unclear.

### Open questions

- **Judge robustness on low-coh outputs.** The same gibberish text gets
  rated 99/100 on both hallucinating and apathetic. Re-judging with a
  prompt that requires coherence > some threshold before assigning a
  trait score would test whether the synergy phenomenon survives.
- **Pair set construction.** Our 28 pairs were not built to break the
  cos–magnitude correlation. A pair set with cos and single-trait
  magnitude orthogonal would let us cleanly separate the two.
- **The mechanical formula's slope under per_axis.** Predicts supp_mean
  ≈ 0 for all pairs. Observed: mean supp_mean = +0.69 unfiltered, +0.09
  at coh ≥ 50 (n = 7). With more pairs surviving coh filter, we could
  test the mean = 0 prediction more definitively.
- **Trait-identity mechanism.** Why apathetic and hallucinating
  dominate is currently a black box. Rubric overlap, baseline
  saturation, semantic class — all hypotheses, none tested.

### Possible next steps (no priority assigned)

- **Audit** the LLM-judge prompt and re-judge the most-collapsed pairs
  to test whether trait ratings on gibberish text reflect the rubric
  intent.
- **Reduce α** under per_axis (α=3) to lower the coh-collapse rate on
  antipodals; this would let us evaluate per_axis with fewer compromised
  pairs.
- **Multi-α scan** under each injection scheme to map the saturation
  envelope; would also test the linear-response assumption.
- **Pair-set design** for a future experiment: select trait pairs to
  decorrelate cos and single-trait magnitude, allowing clean partial
  correlation tests.
- **Trajectory analysis**: under per_axis, what does the hidden-state
  trajectory at L=17 look like for joint vs single, on coh-preserved
  vs coh-collapsed pairs? May illuminate the mechanism.